In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score


In [2]:
df = pd.read_csv(r"D:\CHIRP\analysis\outputs\features_04_parameterTest.csv")
df

,Number,FileName,Valence,Arousal,SocialEngagement,Time,Behavior,Gender,Age,mfcc1_mean,...,chroma6,chroma7,chroma8,chroma9,chroma10,chroma11,chroma12,zcr,rms,spectral_centroid
0,1,Aster_alarmrespondingtohuman_1524,-1.60,3.8,-2.70,1524,respondingtohuman,M,1.5,-390.488342,...,0.185233,0.209743,0.250518,0.223257,0.470626,0.590350,0.657865,0.311491,0.064246,4358.639553
1,2,Aster_alertoncurtain_1906,0.40,1.9,-0.05,1906,alert,M,1.5,-433.151917,...,0.281970,0.347086,0.425936,0.432686,0.381827,0.646774,0.564646,0.230179,0.048830,3505.416980
2,3,Aster_aloneresponding_1458,0.15,0.9,0.00,1458,responding,M,1.5,-337.110504,...,0.145383,0.105791,0.186186,0.427773,0.271388,0.222277,0.615558,0.329690,0.086895,4209.269147
3,4,Aster_alonetalking_1335,0.75,1.0,0.65,1335,selftalk,M,1.5,-417.492706,...,0.283984,0.336734,0.373847,0.252005,0.427924,0.477983,0.482706,0.259977,0.055223,3482.508807
4,5,Aster_angryathomeoccupied_1904,-1.05,4.0,-1.55,1904,angry,M,1.5,-410.802246,...,0.536215,0.519460,0.490378,0.431585,0.375488,0.306628,0.314257,0.231971,0.032142,3584.103029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,312,Fizzy_scratchingaster_1659,3.95,2.2,4.35,1659,allogroom,F,1.0,-453.329071,...,0.491355,0.461562,0.516075,0.573383,0.490576,0.536580,0.802170,0.107699,0.016766,2115.950399
312,313,Fizzy_sleepy_1200,-0.60,3.6,0.00,1200,sleepy,F,1.0,-325.722656,...,0.633860,0.584347,0.494283,0.428000,0.399939,0.332801,0.346417,0.210765,0.025693,3482.031936
313,314,Fizzy_talkingrespondingtoherself_1524,0.75,1.0,0.65,1524,selftalk,F,1.0,-373.357300,...,0.504699,0.528834,0.655875,0.572965,0.394461,0.395497,0.428561,0.387366,0.057758,4816.822092
314,315,Fizzy_talkingtomateandhuman_1323,2.85,3.1,2.60,1323,conversation,F,1.0,-579.687927,...,0.492729,0.771696,0.740513,0.665613,0.457850,0.374878,0.415552,0.173340,0.015644,2604.357468


In [3]:
y = df[["Valence", "Arousal", "SocialEngagement"]]

non_feature_cols = [
    "Number", "FileName",
    "Valence", "Arousal", "SocialEngagement", "Behavior", "Gender"
]

X = df.drop(columns=non_feature_cols)

In [4]:
print(y["Arousal"].describe())

count    316.000000
mean       2.789873
std        1.087516
min        0.000000
25%        2.100000
50%        2.900000
75%        3.600000
max        5.000000
Name: Arousal, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## Lasso Linear Regression

In [6]:
from sklearn.linear_model import Lasso
from scipy.stats import spearmanr
import numpy as np

In [7]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.1))
])

for dim in y.columns:
    print(f"--- {dim} ---")
    pipeline.fit(X_train, y_train[dim])
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test[dim], y_pred)
    r2 = r2_score(y_test[dim], y_pred)
     # Spearman
    spearman_corr, _ = spearmanr(y_test[dim], y_pred)
    
    # CCC
    y_true = y_test[dim].values
    y_pred_np = np.array(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred_np)

    var_true = np.var(y_true)
    var_pred = np.var(y_pred_np)

    cov = np.mean((y_true - mean_true) * (y_pred_np - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    
    print(f"MAE = {mae:.3f}, R² = {r2:.3f}")
    print(f"Spearman = {spearman_corr:.3f}, CCC = {ccc:.3f}\n")


--- Valence ---
MAE = 1.327, R² = 0.063
Spearman = 0.247, CCC = 0.194

--- Arousal ---
MAE = 0.814, R² = 0.012
Spearman = 0.227, CCC = 0.118

--- SocialEngagement ---
MAE = 1.531, R² = 0.062
Spearman = 0.260, CCC = 0.210



## Baseline Model Summary & Reflection